In [17]:
import time
import numpy as np

---

# **Exercise 1**

# Question 1(a): 

# ***Integer Programming Formulation :***

### ***Decision Variables :***

Let $x_i$ be a binary:
$$
x_i = \begin{cases} 1 & \text{if item } i \text{ is selected}\\ 0 & \text{otherwise} \end{cases}
$$

$$i = 1, \dots, 20$$

### ***Objective Function (Maximize Value) :***

$$
\max Z = \sum_{i=1}^{20} v_i x_i 
$$

### ***Constraint :***

$$
\sum_{i=1}^{20} w_{i}x_{i} \le 2400 \quad (40 \text{ hours} = 2400 \text{ minutes})
$$

In [18]:
values = [10,25,33,53,15,70,52,45,32,30,20,5,17,50,37,18,71,35,60,90]
weights = [180,180,150,150,130,210,160,175,90,120,110,60,145,180,120,120,300,300,240,500]

n = len(values)
capacity = 2400

In [19]:
import pulp

start_ip = time.time()

model = pulp.LpProblem("Knapsack", pulp.LpMaximize)
x = [pulp.LpVariable(f"x_{i}", cat='Binary') for i in range(n)]

model += pulp.lpSum(values[i]*x[i] for i in range(n))
model += pulp.lpSum(weights[i]*x[i] for i in range(n)) <= capacity

model.solve(pulp.PULP_CBC_CMD(msg=0))

selected = [i+1 for i in range(n) if x[i].value() == 1]
total_value = sum(values[i-1] for i in selected)

end_ip = time.time()

In [20]:
print("Optimal Value (Integer Programming):", total_value)
print("Selected Tasks:", selected)
print(f"Time Taken: {(end_ip-start_ip)*1000} ms")

Optimal Value (Integer Programming): 623
Selected Tasks: [3, 4, 6, 7, 8, 9, 10, 14, 15, 17, 19, 20]
Time Taken: 9.154558181762695 ms


# Question 1(b): 

# ***Recursion (Brute Force) :***

### ***Logic :***
* At each step: include or exclude item
* Explore full tree

In [ ]:
def knapsack_recursive(i, remaining):
    if i == n or remaining == 0:
        return 0, []   # (value, selected_items)
    
    # Exclude if weight of item exceeds remaining capacity
    if weights[i] > remaining:  
        return knapsack_recursive(i+1, remaining)
    
    # Branch 1 ... Include
    val_incl, items_incl = knapsack_recursive(i+1, remaining-weights[i])
    val_incl += values[i]
    
    # Branch 2 ... Exclude
    val_excl, items_excl = knapsack_recursive(i+1, remaining)
    
    # Select Best Branch 
    if val_incl > val_excl:
        return val_incl, items_incl + [i+1]  # store task number
    else:
        return val_excl, items_excl

In [22]:
start_recursive = time.time() # start
opt_value, selected_tasks = knapsack_recursive(0, capacity)
end_recursive = time.time()   # end

In [23]:
print("Optimal Value (Recursive):", opt_value)
print("Selected Tasks:", sorted(selected_tasks))
print(f"Time Taken: {(end_recursive-start_recursive)*1000} ms")

Optimal Value (Recursive): 623
Selected Tasks: [3, 4, 6, 7, 8, 9, 10, 14, 15, 17, 19, 20]
Time Taken: 234.2367172241211 ms


# Question 1(c): 

# ***Dynamic Programming :***

### ***Logic :***
* Use DP table
* Bottom-up approach

In [ ]:
start_dp = time.time()  # start

dp = np.zeros((n+1, capacity+1))

for i in range(1, n+1):
    for w in range(capacity+1):
        # Exclude or Include weights[i-1] if it does not exceeds capacity
        if weights[i-1] <= w:
            dp[i][w] = max(dp[i-1][w],
                            values[i-1] + dp[i-1][w - weights[i-1]])
        else:
            # Exclude weights[i-1] if it exceeds capacity
            dp[i][w] = dp[i-1][w]   # copy from above cells

# Backtracking
w = capacity
selected = []
for i in range(n, 0, -1):
    if dp[i][w] != dp[i-1][w]:
        selected.append(i)
        w -= weights[i-1]

end_dp = time.time()    # end 

In [25]:
print("Optimal Value (DP):", int(dp[n][capacity]))
print("Selected Tasks:", selected[::-1])
print(f"Time Taken: {(end_dp-start_dp)*1000} ms")

Optimal Value (DP): 623
Selected Tasks: [3, 4, 6, 7, 8, 9, 10, 14, 15, 17, 19, 20]
Time Taken: 40.26293754577637 ms


### ***Verdict :***

* ***The recursive approach is computationally expensive due to its exponential complexity.***
* ***Dynamic Programming significantly reduces computation time using memoization.***
* ***The Integer Programming solver performs best in practice due to highly optimized algorithms.***

# Question 2: 

# ***Greedy Method by Value :***

In [26]:
start_greedy = time.time()

# Sort tasks by value (descending)
items = list(range(n))
items.sort(key=lambda i: values[i], reverse=True)

total_w = 0
total_v = 0
selected = []

for i in items:
    if total_w + weights[i] <= capacity:
        selected.append(i+1)
        total_w += weights[i]
        total_v += values[i]

end_greedy = time.time()

In [27]:
print("Selected Tasks:", selected)
print("Total Value:", total_v)
print("Total Time (<=2400):", total_w)
print("Time Taken:", (end_greedy-start_greedy)*1000, "ms")

Selected Tasks: [20, 17, 6, 19, 4, 7, 14, 8, 15, 18, 12]
Total Value: 568
Total Time (<=2400): 2395
Time Taken: 0.2503395080566406 ms


In this approach, the manager selects tasks in descending order of value and includes them as long as the total time does not exceed the available limit of 2400 minutes (40 hours).

The greedy method prioritizes tasks with the highest individual values without considering their durations relative to other tasks. As a result, it may include high-value but time-consuming tasks early, which prevents inclusion of better combinations of smaller tasks.

In this case:

* Greedy Value = **568**
* Optimal Value = **623**

Thus, the greedy approach produces a suboptimal solution.

# Question 3: 

# ***Greedy Method by Value/Weight ratio :***

In [28]:
start_greedy_ratio = time.time()

# Sort by value/weight ratio (descending)
items = list(range(n))
items.sort(key=lambda i: values[i]/weights[i], reverse=True)

total_w = 0
total_v = 0
selected = []

for i in items:
    if total_w + weights[i] <= capacity:
        selected.append(i+1)
        total_w += weights[i]
        total_v += values[i]

end_greedy_ratio = time.time()

In [29]:
print("Selected Tasks:", selected)
print("Total Value:", total_v)
print("Total Time (<=2400):", total_w)
print("Time Taken:", (end_greedy_ratio - start_greedy_ratio)*1000, "ms")

Selected Tasks: [9, 4, 6, 7, 15, 14, 8, 10, 19, 17, 3, 11, 16, 2, 12]
Total Value: 601
Total Time (<=2400): 2365
Time Taken: 0.2129077911376953 ms


In this approach, the manager selects tasks in descending order of value-to-time ratio and includes them as long as the total time does not exceed the available limit of 2400 minutes (40 hours).

The greedy method prioritizes tasks that provide the highest value per unit time. While this considers efficiency better than selecting by value alone, it still does not account for the overall combination of tasks. As a result, it may exclude certain combinations that yield a higher total value.

In this case:

* Greedy (Ratio) Value = **601**
* Optimal Value = **623**

Thus, the greedy approach produces a suboptimal solution, although it performs better than the value-based greedy method.

# Question 4: 

# ***Problem :***
Consider the sets:
$$
K := \{x \in \{0, 1\}^n : \sum_{i=1}^n a_i x_i \le b\}
$$
$$
K_C := \{x \in \{0, 1\}^n : \sum_{i \in C} x_i \le |C| - 1, \forall \text{ minimal covers } C\}
$$
Determine whether:
$K \subseteq K_C$, $K_C \subseteq K$, both, or neither

# ***Analysis :***

A minimal cover $C$ is a subset of items such that:
$$
\sum_{i \in C} a_i > b
$$
and removing any element from $C$ makes it feasible.

### **(i) Showing $K \subseteq K_C$ :**
Let $x \in K$, i.e.,$$\sum_{i=1}^n a_i x_i \le b$$If for some minimal cover $C$, all variables $x_i = 1$ for $i \in C$, then:$$\sum_{i \in C} a_i > b$$which violates the knapsack constraint. Hence, at least one variable in $C$ must be zero, implying:$$\sum_{i \in C} x_i \le |C| - 1$$Therefore, every feasible solution in $K$ satisfies all cover inequalities.$$\Rightarrow K \subseteq K_C$$

### **(ii) Checking whether $K_C \subseteq K$ :**
A vector $x \in K_C$ satisfies all cover inequalities, but this does not guarantee:$$\sum_{i=1}^n a_i x_i \le b$$This is because cover inequalities restrict only specific subsets of variables and do not fully enforce the overall capacity constraint. Thus, it is possible for a solution to satisfy all cover inequalities while still violating the knapsack constraint.$$\Rightarrow K_C \not\subseteq K$$

### **Final Result :**
$K \subseteq K_C$ but $K_C \not\subseteq K$

### **Conclusion :**
The set of feasible solutions of the knapsack problem is contained within the set defined by minimal cover inequalities. However, satisfying all cover inequalities does not ensure feasibility with respect to the original knapsack constraint. Therefore, $K \subseteq K_C$, but $K_C \not\subseteq K$.

---

# **Exercise 2**

# Question 1: 

# ***Integer Programming Formulation :***

### ***Decision Variables :***

Let:
* $x_{ij}$: number of boxes of type $j$ used to satisfy demand of type $i$
* $y_j \in \{0, 1\}$: 1 if box type $j$ is used, else 0

(Only allow $j \ge i$ since larger boxes can satisfy smaller demand)

### ***Parameters :***

* Demand: $d_i$
* Volume: $v_j$
* Variable cost: $100 \cdot v_j$
* Fixed cost: 150

### ***Objective Function (Minimize Cost) :***

$$
\min \sum_{j=1}^{6} \left( 100 \cdot v_j \cdot \sum_{i \le j} x_{ij} \right) + 150 \cdot y_j
$$

### **Constraints :**


**1. Demand Satisfaction**
$$\sum_{j \ge i} x_{ij} = d_i \quad \forall i$$

**2. Activation Constraint**
$$\sum_{i} x_{ij} \le M \cdot y_j \quad \forall j$$

**3. Non-negativity**
$$x_{ij} \ge 0, \quad y_j \in \{0, 1\}$$

# Question 2: 

In [30]:
demand = [300,450,80,500,150,800]
volume = [0.15,0.25,0.35,0.5,0.6,0.75]
price = [300,300,350,500,600,800]

var_cost = [100*v for v in volume]
n = len(demand)

In [ ]:
x = np.zeros((n,n), dtype=int)
used = [0]*n  # track if box type used

total_cost = 0

for i in range(n):
    remaining = demand[i]
    
    for j in range(i, n):  # j >= i
        if remaining > 0:
            x[i][j] = remaining
            total_cost += remaining * var_cost[j]
            used[j] = 1
            remaining = 0

# add fixed cost once per used box type
total_cost += sum(150*used[j] for j in range(n))

print("Allocation Matrix:\n", x)
print("Total Cost:", total_cost)

Allocation Matrix:
 [[300   0   0   0   0   0]
 [  0 450   0   0   0   0]
 [  0   0  80   0   0   0]
 [  0   0   0 500   0   0]
 [  0   0   0   0 150   0]
 [  0   0   0   0   0 800]]
Total Cost: 113450.0


# Question 3: 

### ***Problem Description :***

The company now has a transport capacity of 1000 cubic meters, so it may not be able to satisfy full demand. The goal is to maximize profit by selecting how many boxes of each type to transport.

### ***Decision Variables :***
Let:
$x_j =$ number of boxes of type $j, \quad j = 1, \dots, 6$

### ***Parameters :***
* $d_j$: demand for box type $j$
* $v_j$: volume of box type $j$
* $p_j$: selling price of box type $j$
Variable cost per box $= 100 \cdot v_j$

### ***Objective Function (Maximize Profit) :***

$$
\max \sum_{j=1}^{6} [(p_j - 100 \cdot v_j) x_j - 150y_j]
$$
***Profit = Selling Price – Production Cost***

### ***Constraints :***

1. Capacity Constraint: $$\sum_{j=1}^{6} v_j \cdot x_j \le 1000$$
2. Demand Constraints: $$0 \le x_j \le d_j \quad \forall j = 1, \dots, 6$$
3. Linking Constraint: $$x_j \le M \cdot y_j \quad \forall j$$
4. Integrality: $$x_j \in \mathbb{Z}_{\ge 0}, \quad y_j \in \{0, 1\}$$

### ***Final Model :***

$$\max \sum_{j=1}^{6} [(p_j - 100 \cdot v_j) x_j - 150y_j]$$
$$\text{s.t. } \sum_{j=1}^{6} v_j x_j \le 1000$$
$$0 \le x_j \le d_j \quad \forall j$$
$$x_j \le M \cdot y_j \quad \forall j$$
$$x_j \in \mathbb{Z}_{\ge 0}, \quad y_j \in \{0, 1\}$$

# Question 4: 

In [ ]:
import pulp

# Model
model = pulp.LpProblem("Profit_Max", pulp.LpMaximize)

# Decision variables
x = [pulp.LpVariable(f"x_{i}", lowBound=0, cat='Integer') for i in range(n)]
y = [pulp.LpVariable(f"y_{i}", cat='Binary') for i in range(n)]

# Objective (profit - fixed cost)
model += pulp.lpSum((price[i]-100*volume[i])*x[i] - 150*y[i] for i in range(n)) 

# Capacity constraint
model += pulp.lpSum(volume[i]*x[i] for i in range(n)) <= 1000

# Demand constraints
for i in range(n):
    model += x[i] <= demand[i]

# Activation constraint (VERY IMPORTANT)
M = max(demand)
for i in range(n):
    model += x[i] <= M * y[i]

# Solve
model.solve(pulp.PULP_CBC_CMD(msg=0))

# Output
print("Optimal Profit:", pulp.value(model.objective))

for i in range(n):
    print(f"Box {i+1}:", x[i].value())

Optimal Profit: 1006900.0
Box 1: 300.0
Box 2: 450.0
Box 3: 0.0
Box 4: 485.0
Box 5: 0.0
Box 6: 800.0


---